In [ ]:
import pandas as pd
import scipy.stats as st
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('Данные.csv', sep = ';')
df.head()

In [ ]:
df.info() # есть строки без id_client, они не дадут информации, лучше удалить 

In [ ]:
df = df[df['id_client'].notna()]

In [ ]:
df['id_client'] = df['id_client'].astype(int) # приводим к нужным форматам
df['dtime_pay'] = pd.to_datetime(df['dtime_pay'],dayfirst = True)
df.head()

In [ ]:
df.info()

In [ ]:
cl = pd.read_csv('Clients.csv', sep = ';')
cl.head()

In [ ]:
cl.info() # есть пустые строки с dtime_ad, что значит, нет информации о показе рекламы. Эти данные лучше удалить.

In [ ]:
cl = cl[cl['dtime_ad'].notna()]

In [ ]:
cl.info() # теперь с этими данными можно работать

In [ ]:
rd = pd.read_csv('Region_dict.csv', sep = ';', encoding = 'cp1251')
rd # импортировалсоь много пустых строк, нужно от них избавитсься 

In [ ]:
rd.info()

In [ ]:
rd = rd.dropna().copy()
rd['id_trading_point'] = rd['id_trading_point'].astype(int)

In [ ]:
rd.info()

In [ ]:
city_count = rd.groupby('city').agg(point_cnt = ('id_trading_point','count')).reset_index().sort_values('point_cnt', ascending = False)
city_count

In [ ]:
plt.figure(figsize = (16,6))
plt.bar(city_count['city'], city_count['point_cnt'])
plt.ylabel('Количество торговых точек')
plt.xticks(rotation = 45, ha = 'right')
plt.title('Количество торговых точек по городам')
plt.show()

In [ ]:
client_pay = df.groupby('id_client').agg(amt_pay = ('amt_payment', 'sum')).reset_index()
client_pay.head()                               

In [ ]:
df_cl = cl.merge(client_pay, on = 'id_client', how = 'left')
df_cl.head()

In [ ]:
df_cl['amt_pay'] = df_cl['amt_pay'].fillna(0)
df_cl.head()

In [ ]:
df_cl_reg = df_cl.merge(rd, on = 'id_trading_point', how = 'left')
df_cl_reg.head()

In [ ]:
df_cl_reg['flag_pay'] = np.where(df_cl_reg['amt_pay'] > 0, 1, 0)
df_cl_reg.head()

Проверим, не пересекаются ли пользователи в тестовой и контрольной группах

In [ ]:
test_users = df_cl_reg[df_cl_reg['nflag_test'] == 1]['id_client']
test_users.head()

In [ ]:
ctrl_users = df_cl_reg[df_cl_reg['nflag_test'] == 0]['id_client']
ctrl_users.head()

In [ ]:
bad_ids = set(test_users) & set(ctrl_users) # таких пользователей нет 
bad_ids

Создаем функцию для ttest 

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
def test_calc(r1, r2, alpha = 0.05):
    t_stat, p_value = ttest_ind(r1, r2)

    print("t-статистика:", t_stat)
    print("p_value:", p_value)

    if p_value < alpha:
        print("Есть статистически значимая разница")
    else:
        print("Нет статистически значимой разницы")  

Создаем функцию  mann_whitney

In [ ]:
from scipy.stats import mannwhitneyu

In [ ]:
def mann_whitney_func(r1, r2, alpha = 0.05):
    stat, p_value = mannwhitneyu(r1, r2)
    
    print("Статистика Манна-Уитни:", stat)
    print("p_value:", p_value)

    if p_value < alpha:
        print("Есть статистически значимая разница")
    else:
        print("Нет статистически значимой разницы")    

Находим "плохие" торговые точки

In [ ]:
test_users = df_cl_reg[df_cl_reg['nflag_test'] == 1]
test_users.head()

In [ ]:
trading_points_test = test_users.groupby('id_trading_point').agg(amt = ('amt_pay', 'sum')).reset_index()
trading_points_test.head()

In [ ]:
bad_points_test = trading_points_test[trading_points_test['amt'] == 0]['id_trading_point'].tolist()
bad_points_test

In [ ]:
ctrl_users = df_cl_reg[df_cl_reg['nflag_test'] == 0]
ctrl_users.head()

In [ ]:
trading_points_ctrl = ctrl_users.groupby('id_trading_point').agg(amt = ('amt_pay', 'sum')).reset_index()
trading_points_ctrl.head()

In [ ]:
bad_points_ctrl = trading_points_ctrl[trading_points_ctrl['amt'] == 0]['id_trading_point'].tolist()
bad_points_ctrl

In [ ]:
bad_points_all= set(bad_points_ctrl) | set(bad_points_test)
bad_points_list = sorted(list(bad_points_all))
bad_points_list

Теперь циклом:

In [ ]:
bad_points = []

for i in df_cl_reg['id_trading_point'].unique():
    point = df_cl_reg[df_cl_reg['id_trading_point'] == i]
    
    test = point[point['nflag_test'] == 1]
    ctrl = point[point['nflag_test'] == 0]
    
    test_sum = test['amt_pay'].sum()
    ctrl_sum = ctrl['amt_pay'].sum()

    if test_sum == 0 or ctrl_sum == 0:
        bad_points.append(i)
        
sorted(bad_points)

Результат другой, потому что .sum() игнорирует Nan (получаются нули). Захватил Случай 1, когда группа есть, а никто не заплатил, и Случай 2, когда группы вообще нет (нет клиентов группе тест и сумма test_sum == 0). 

In [ ]:
points_test = test_users.groupby('id_trading_point').agg(cnt_test = ('id_client', 'count')).reset_index()
points_ctrl = ctrl_users.groupby('id_trading_point').agg(cnt_test = ('id_client', 'count')).reset_index()

In [ ]:
test_points = points_test['id_trading_point'].tolist()
ctrl_points = points_ctrl['id_trading_point'].tolist()

In [ ]:
empt_points = list(set(test_points) ^ set(ctrl_points))
sorted(empt_points)

Теперь циклом:

In [ ]:
empty_points = []

for i in df_cl_reg['id_trading_point'].unique():
    point = df_cl_reg[df_cl_reg['id_trading_point'] == i]
    
    test = point[point['nflag_test'] == 1]
    ctrl = point[point['nflag_test'] == 0]
    
    test_count = len(test)
    ctrl_count = len(ctrl)

    if test_count == 0 or ctrl_count == 0:
        empty_points.append(i)
        
sorted(empty_points)

Получается, что первый цикл нашел все "плохие" точки.

Расчет результатов

In [ ]:
all_b_ps = bad_points_list + empt_points
all_b_ps

In [ ]:
df_final = df_cl_reg[~df_cl_reg['id_trading_point'].isin(all_b_ps)] # удалили эти торговые точки
df_final.head()

In [ ]:
test_pay = point[point['nflag_test'] == 1]['amt_pay']
ctrl_pay = point[point['nflag_test'] == 0]['amt_pay']

plt.hist(test_pay, bins = 30, alpha = 0.5, label = 'Test')
plt.hist(ctrl_pay, bins = 30, alpha = 0.5, label = 'Control')
plt.title('Распределение платежей')
plt.xlabel('Сумма платежей')
plt.ylabel('Количество платежей')
plt.legend()
plt.show()

Считаем конверсии и чеки: 

In [ ]:
test_f = df_final[df_final['nflag_test'] == 1]
ctrl_f = df_final[df_final['nflag_test'] == 0]

In [ ]:
conv_test = test_f['flag_pay'].mean()
conv_ctrl = ctrl_f['flag_pay'].mean()
print(conv_test)
print(conv_ctrl) 

In [ ]:
avg_check_test = round(test_f ['amt_pay'].mean(), 2)
avg_check_ctrl = round(ctrl_f['amt_pay'].mean(), 2)
print(avg_check_test)
print(avg_check_ctrl)

Получаем, что конверсия в тестовой группе больше и средний чек больше. Теперь проверим эти данные на статистическую значимость. 

In [ ]:
test_calc(test_f['flag_pay'], ctrl_f['flag_pay']) # тестируем конверсии 

In [ ]:
mann_whitney_func(test_f['flag_pay'], ctrl_f['flag_pay'])

In [ ]:
test_calc(test_f ['amt_pay'], ctrl_f['amt_pay']) # тестиреум чеки

In [ ]:
mann_whitney_func(test_f['amt_pay'], ctrl_f['amt_pay'])

Данные подтверждаются, есть статистически значимая разница во всех тестах. 

Теперь сделаем сегментацию по городам. 

In [ ]:
for city in df_final['city'].unique():
    df_city = df_final[df_final['city'] == city]

    test = df_city[df_city['nflag_test'] == 1]
    ctrl = df_city[df_city['nflag_test'] == 0]

    conv_test = test['flag_pay'].mean()
    conv_ctrl = ctrl['flag_pay'].mean()

    check_test = test['amt_pay'].mean()
    check_ctrl = ctrl['amt_pay'].mean()

    print(city)
    print('Конвесрия test:', conv_test)
    print('Конвесрия ctrl:', conv_ctrl)
    print('Чек test:', check_test)
    print('Чек ctrl:', check_ctrl)
    print('____________')

Во всех городах в тестовой группе выще и чек, и конверсия, кроме Сочи и Краснодара (чек и конверсия ниже).

Статистическая проверка t-test

In [ ]:
for city in df_final['city'].unique():
    df_city = df_final[df_final['city'] == city]

    test = df_city[df_city['nflag_test'] == 1]
    ctrl = df_city[df_city['nflag_test'] == 0]

    print(city)
    print('T-test конверсия:')
    test_calc(test['flag_pay'], ctrl['flag_pay'])

    print('T-test чек:')
    test_calc(test['amt_pay'], ctrl['amt_pay'])

    print('____________')

Проверку t-test прошли Санкт-Петербург, Москва, Волгоград, Владимир. 

In [ ]:
for city in df_final['city'].unique():
    df_city = df_final[df_final['city'] == city]

    test = df_city[df_city['nflag_test'] == 1]
    ctrl = df_city[df_city['nflag_test'] == 0]

    print(city)
    print('Mann-Whitnye конверсия:')
    mann_whitney_func(test['flag_pay'], ctrl['flag_pay'])

    print('Mann-Whitnye чек:')
    mann_whitney_func(test['amt_pay'], ctrl['amt_pay'])

    print('____________')

Проверку Манна-Уитни прошли Санкт-Петербург, Москва, Тюмень, Волгоград, Владимир, Самара. 

Вывод: Оба теста показали статистическую разницу для Санкт-Петербурга, Москвы, Волгограда и Владимира. Здесь можно пропобвать вводить тест. 
Тюмнень и Самара прошли ранговый тест Манна-Уитни, то есть люди платят чаще, можно побробовать тест, если конверсия важнее. 

Создаем финальную таблицу для выгрузки в Excel

In [ ]:
base_stats = df_final.groupby(['city','id_trading_point', 'nflag_test']).agg(
    count = ('amt_pay', 'count'),
    avg_payment = ('amt_pay', 'mean'),
    sigma = ('amt_pay', 'std')).reset_index()
base_stats.head()

In [ ]:
pivot = base_stats.pivot_table(
    index = ['city', 'id_trading_point'],
    columns = 'nflag_test',
    values = ['count', 'avg_payment', 'sigma']).reset_index()

pivot.head()

In [ ]:
pivot.columns.name = None

In [ ]:
pivot.columns = ['city', 'id_trading_point', 'avg_payment_control', 'avg_payment_test', 'count_control', 'count_test', 'sigma_control', 'sigma_test']
pivot.head()

In [ ]:
pivot['count_all'] = pivot['count_test'] + pivot['count_control']
pivot.head()

In [ ]:
pivot['percent_count'] = pivot['count_all'] / pivot['count_all'].sum()
pivot['diff'] = pivot['avg_payment_test'] - pivot['avg_payment_control']
pivot.head()

In [ ]:
ttest_list = []
pvalue_list = []

for i in range(len(pivot)):

    city = pivot['city'][i]
    point = pivot['id_trading_point'][i]

    df_point = df_final[(df_final['city'] == city) & (df_final['id_trading_point'] == point)]

    test = df_point[df_point['nflag_test'] == 1]['amt_pay']
    ctrl = df_point[df_point['nflag_test'] == 0]['amt_pay']

    t_tsat, p_val = ttest_ind(test, ctrl)

    ttest_list.append(t_tsat)
    pvalue_list.append(p_val)

In [ ]:
pivot['ttest'] = ttest_list
pivot['pvalue_ttest'] = pvalue_list
pivot.head()

In [ ]:
labels = []

for i in range(len(pivot)):
    p = pivot['pvalue_ttest'][i]
    diff = pivot['diff'][i]

    if p < 0.05 and diff > 0:
        labels.append('positive')
    elif p < 0.05 and diff < 0:
        labels.append('negative')
    else:
        labels.append('neutral')


pivot['labels'] = labels

In [ ]:
pivot.head()

In [ ]:
with pd.ExcelWriter('result.xlsx') as writer:

    pivot.to_excel(writer, sheet_name = 'all_data', index = False)
    
    df_pos = pivot[pivot['labels'] == 'positive']
    df_neg = pivot[pivot['labels'] == 'negative']
    df_neu = pivot[pivot['labels'] == 'neutral']

    if len(df_pos) > 0:
        df_pos.to_excel(writer, sheet_name = 'positive', index = False)

    if len(df_neg) > 0:
        df_neg.to_excel(writer, sheet_name = 'negative', index = False)

    if len(df_neu) > 0:
        df_neu.to_excel(writer, sheet_name = 'neutral', index = False)